# Minecraftism - train your own uncensored Minecraft + everyday LLM (Kaggle, GPU T4 x2)

**Minecraftism** is a small open model fine-tuned into **an everyday assistant that knows Minecraft inside out** - and,
inside [PyMC Bot](https://github.com/codero-sus/PyMC_Bot), the brain that plays the game (JSON actions, chat, mob facts).

| Edition | Base (uncensored = abliterated copy by huihui-ai) | What it does |
| --- | --- | --- |
| `minecraftism` | Qwen3-4B-Instruct-2507 | text, answers straight away - **recommended** |
| `minecraftism-thinking` | Qwen3-4B-Thinking-2507 | text, thinks first (`<think>`), better at tricky questions, slower |
| `minecraftism-vl` | Qwen3-VL-4B-Instruct | also reads pictures (your screenshots and/or streamed ones) |
| `minecraftism-vl-thinking` | Qwen3-VL-4B-Thinking | pictures + thinking (slowest) |

**Uncensored** (`UNCENSORED = True`): trains on the *abliterated* base (the refusal direction removed from the weights),
drops refusals / "as an AI language model" answers from the everyday chat data and adds frank example answers, so it
answers directly instead of lecturing. The files get `-Uncensored` in their names.

**Commercial-use data** (`COMMERCIAL_ONLY = True`, the default): every training source allows commercial use -
everyday chat from OpenAssistant (Apache-2.0), `<think>` traces from NVIDIA Nemotron (CC BY 4.0), Minecraft Q&A
generated from PrismarineJS/minecraft-data (MIT) and, for the VL editions, Minecraft frames from
shinohararin/minecraft-raytrace (MIT) with exact questions and answers made from their raycast labels. The kit's
Minecraft Wiki rows (CC BY-NC-SA) are left out. The model card lists every source with its licence.
`COMMERCIAL_ONLY = False` uses the bigger non-commercial sets instead (FineTome, naklecha's 695k Minecraft Q&A,
CraftJarvis captioned screenshots, Minecraft Wiki text).

**Big data without big downloads:** the notebook streams everything external - only the rows and pictures that are
used are downloaded (minecraft-raytrace is read one ~330 MB shard at a time, without its voxel dumps; the 106 GB JARVIS set is
not used: it teaches raw keyboard/mouse tokens, not answers).

**What this notebook does**, per edition: dry-run check -> download extra everyday chat -> QLoRA training -> eval on held-out
rows -> *optional* test suite -> safetensors export (compacted: 16-bit, tied embeddings, so a 4B model stays 4B-sized)
-> **upload the safetensors** -> GGUFs made from them with llama.cpp (Q8_0, Q6_K, Q5_K_M, Q4_K_M, Q3_K_M), **each
uploaded as soon as it is ready** -> *optional* GGUF test in Ollama -> GGUF repo (Modelfile, card, `tokenizer.json` and
configs) -> clean up.

**Two editions of every model** go to Hugging Face, each in its own repo:

| Repo (e.g.) | Format | For |
| --- | --- | --- |
| `you/Minecraftism-4B-Uncensored-GGUF` | GGUF `Q8_0` (~4.3 GB), `Q6_K`, `Q5_K_M`, `Q4_K_M` (~2.5 GB, default), `Q3_K_M` (+ `mmproj` for VL), `Modelfile`, `tokenizer.json` + configs | Ollama, llama.cpp, LM Studio, **PyMC Bot** |
| `you/Minecraftism-4B-Uncensored` | **safetensors**, the LoRA merged into the base in 16-bit (~8 GB for 4B params), `tokenizer.json` + configs | transformers, vLLM, further fine-tuning, your own quantisations |

Why ~8 GB for the safetensors? Every one of the 4 billion weights takes 2 bytes in 16-bit. The trainer compacts the
merge (no float32 leftovers, no duplicate output matrix), so it is exactly the base model's size and never bigger; the
small files are the GGUF quants (Q4_K_M keeps ~4.9 bits per weight).

**Before you press Run All**
1. On your PC: `python -m pymc_bot llm build` (VL editions: add `--playtime <mod folder> --screenshots`, recorded with
   `/pymcbot screenshots on`) and upload the `llm_data` folder as a Kaggle dataset (with `images/`, without `wiki_cache`).
   No PC at hand? Skip this: with no dataset added, the notebook downloads the ready-made kit
   (`pymc-minecraft-llm-kit.zip`) from the newest PyMC Bot GitHub release. An older dataset is fine too: the notebook
   then fetches the newest trainer from the release.
2. Here: **Add Input** -> your dataset. **Settings** -> Accelerator **GPU T4 x2**, Internet **on**.
3. To publish on Hugging Face: **Add-ons -> Secrets -> Add secret** `HF_TOKEN` (a *write* token from
   huggingface.co/settings/tokens) and tick it for this notebook. It is read here, never printed.
4. Edit the settings cell below. First time? Set `QUICK_TEST = True` (about 10-15 min) to check everything, then train for real.

**Speed - both T4s, every minute.** A Kaggle session stops after **12 h** and the weekly GPU quota is ~30 h, so the
notebook never leaves a T4 idle (`GPU_MODE = "auto"`): with two or more editions each T4 trains its own edition at the
same time (two editions in the time of one); with one edition both T4s train it together (DDP, ~1.7-1.9x faster).
`FAST = True` adds length-grouped batches and parallel data loading, and `MAX_HOURS` ends training early enough that
the export and upload always finish before the 12 h cut-off. For long runs use **Save Version -> Save & Run All**: it
runs in the background, keeps `/kaggle/working` as the Output and the session ends by itself when done
(`AUTO_EXIT = True` also ends an interactive session once everything is on Hugging Face).

Made with ❤️ by codero-sus - https://github.com/codero-sus - https://codero.is-a.dev - https://the-codero.web.app/ - personal, non-commercial use (PolyForm Noncommercial 1.0.0).

In [ ]:
# ============================== settings ==============================
# Editions to train in this session, in order:
#   "minecraftism"              text, fast (recommended)        "minecraftism-thinking"     text, thinks first
#   "minecraftism-vl"           reads pictures                  "minecraftism-vl-thinking"  pictures + thinking
# Older text sizes work too: "qwen3-4b", "qwen3-8b", "qwen3-14b" (add --split to EXTRA), "llama-3.2-3b", "qwen3.5-4b".
EDITIONS = ["minecraftism"]

DATA = "auto"            # "auto" = your Kaggle dataset (Add Input) if there is one, else the ready-made kit from the
                         # newest PyMC Bot GitHub release; or a path, e.g. "/kaggle/input/my-llm-data"
TRAINER = "auto"         # "auto" = the trainer in your dataset, or the newest one from GitHub when that one is too old
                         # for these settings; "latest" = always the newest; "dataset" = never download

UNCENSORED = True        # abliterated base + refusal-free chat data (files named ...-Uncensored)
GENERAL_CHAT = 3000      # everyday chat rows mixed in so it stays a great daily assistant (0 = Minecraft only)
                         # (thinking editions get half as many rows with real <think> traces instead)
COMMERCIAL_ONLY = True   # only data whose licence allows commercial use (licences listed in the model card):
                         # OpenAssistant chat (Apache-2.0), NVIDIA Nemotron maths traces (CC BY 4.0), Q&A generated
                         # from minecraft-data (MIT), minecraft-raytrace pictures (MIT); Minecraft Wiki rows left out.
                         # False = the bigger non-commercial sets (FineTome, naklecha 695k Q&A, CraftJarvis, wiki)
MINECRAFT_QA = 20000     # Minecraft Q&A rows. Commercial: exact facts generated from minecraft-data (recipes, tools,
                         # drops, food, enchantments, mobs, biomes - about 10k rows at most). Otherwise streamed from
                         # the 695k-row naklecha/minecraft-question-answer-700k (sampled + filtered). 0 = off
STREAM_PICTURES = 3000   # VL editions: picture rows, streamed so VL trains even without your own screenshots. 0 = off
                         # Commercial: minecraft-raytrace frames, 2 exact Q&A each (biome, time, weather, block at the
                         # crosshair, mobs), read one ~330 MB shard at a time (~3 shards). Otherwise CraftJarvis/
                         # minecraft-vlp captions / Q&A read out of its zips (~150-300 MB)
PERSONA_MIX = 0.5        # share of Minecraft Q&A rows that use SYSTEM_PROMPT (game turns keep the bot's own prompt)

# The model's default personality. It goes into the training data and the Modelfile; Ollama uses it whenever an
# app sends no system prompt of its own (PyMC Bot sends its own short prompt for game turns). Edit freely.
SYSTEM_PROMPT = """You are Minecraftism, an AI assistant made by codero-sus.

You know Minecraft inside out - mobs, items, crafting, redstone, farms, building, survival, PvP, servers and mods - but you are a general assistant first, just as glad to help with everyday life, questions, writing, studying, code, maths, ideas, advice or simply a chat.

How you work:
- Be genuinely useful. Answer what was asked with the details that matter, then stop. No filler.
- Be honest. If you don't know or aren't sure, say so. Never invent facts, numbers, versions or sources.
- Treat the user as a capable adult. Don't lecture, moralise or add warnings and disclaimers nobody asked for, and don't refuse something just because it is edgy, dark, mature or controversial.
- Have a personality: friendly, witty when it fits, with your own opinions when asked.
- Match the user: short and casual for chat, clear and structured for real tasks. Reply in the user's language.
- Use Markdown only when it helps (steps, lists, code).

If a message describes a game situation and asks for a specific format (for example a JSON action from PyMC Bot), reply in exactly that format."""

HF_REPO = "auto"         # "auto" = <your HF name>/<edition repo>, e.g. Minecraftism-4B-Uncensored-GGUF; "" = no upload;
                         # "name/repo" only when training a single edition
HF_PRIVATE = False       # True = private Hugging Face repos
HF_LORA = False          # also upload the LoRA adapter
SAFETENSORS = True       # also export + upload the full-precision safetensors edition (16-bit merged, ~8 GB) to
                         # <repo without -GGUF>, e.g. Minecraftism-4B-Uncensored, next to the GGUF repo
GGUF_QUANTS = "q8_0,q6_k,q5_k_m,q4_k_m,q3_k_m"  # GGUFs made from the safetensors and uploaded one by one to the
                         # GGUF repo (only when uploading; the main one is --gguf, q4_k_m). "" = only q4_k_m

QUICK_TEST = False       # True = 30 steps, small eval, no GGUF/safetensors/upload: checks the whole pipeline in ~10-15 min
RUN_TESTS = True         # after training: bot JSON, Minecraft Q&A, everyday prompts, refusal check, identity, vision
TEST_GGUF = False        # also install Ollama here and chat with the exported GGUF (+ ~3 min per edition)
CLEAN_AFTER_UPLOAD = True  # delete the big files once an edition is safely on Hugging Face (Kaggle has 20 GB)

# ---- speed: Kaggle gives two T4s and ~30 GPU hours a week, so both T4s work all the time
GPU_MODE = "auto"        # "auto": 2+ editions -> one edition per T4 at the same time (twice the editions per hour);
                         #         1 edition -> both T4s train it together (DDP, ~1.7-1.9x faster; one T4 if that fails)
                         # "parallel" / "ddp" force one of those; "single" = one T4, one edition after another (old way)
FAST = True              # max speed: batches of similar length, 2 data-loader workers per GPU, 60 eval rows (not 150)
MAX_HOURS = 11.5         # Kaggle stops a session at 12 h: training ends early enough that export + upload still fit,
                         # so a long run is never lost (an edition that cannot fit any more waits for the next session)
AUTO_EXIT = True         # end the session when everything is done (last cell) so no GPU quota idles away

EPOCHS = 1               # 2 = learns the data better, takes twice as long
EXTRA = ""               # more trainer flags, e.g. "--gguf q5_k_m", "--rank 64", "--lr 1e-4", "--image-side 448"

In [ ]:
%%capture
# Unsloth's Kaggle install (same as their Qwen3 / Qwen3-VL notebooks)
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
# ============================== environment check ==============================
import glob
import json
import os
import shutil
import socket
import subprocess
import time
import urllib.request
import zipfile


def check(ok, good, bad):
    print(("OK    " if ok else "WARN  ") + (good if ok else bad))
    return ok

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
check(bool(gpus), f"GPU: {', '.join(gpus)}", "no GPU - Settings > Accelerator > GPU T4 x2")
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    online = True
except OSError:
    online = False
check(online, "internet is on", "internet is OFF - Settings > Internet on (needed for the models and the chat data)")
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
check(free_gb > 12, f"{free_gb:.0f} GB free in /kaggle/working", f"only {free_gb:.0f} GB free - clean /kaggle/working")
tmp_gb = shutil.disk_usage("/tmp").free / 1e9  # the F16 GGUF + quants are made here, off the 20 GB output disk
check(tmp_gb > 20, f"{tmp_gb:.0f} GB free in /tmp for the GGUF quants", f"only {tmp_gb:.0f} GB free in /tmp")

# ---- training data + trainer: your dataset, else the newest PyMC Bot release on GitHub
RELEASES = "https://api.github.com/repos/codero-sus/PyMC_Bot/releases?per_page=30"

def release_asset(name):
    """Download URL of `name` from the newest PyMC Bot release that has it (None when offline)."""
    try:
        request = urllib.request.Request(RELEASES, headers={"Accept": "application/vnd.github+json"})
        releases = json.load(urllib.request.urlopen(request, timeout=30))
    except Exception as exc:
        print(f"WARN  GitHub releases not reachable ({type(exc).__name__})")
        return None
    for release in sorted(releases, key=lambda r: r.get("published_at") or "", reverse=True):
        for asset in release.get("assets", []):
            if asset["name"] == name:
                return asset["browser_download_url"], release["tag_name"]
    return None

def download(name, target):
    found = release_asset(name)
    if not found:
        return None
    url, tag = found
    urllib.request.urlretrieve(url, target)
    print(f"OK    downloaded {name} from release {tag}")
    return target

if DATA == "auto":
    found = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
    if found:
        DATA = os.path.dirname(found[0])
    else:
        print("no dataset added - using the ready-made Minecraftism kit (no recorded playtime / screenshots in it)")
        kit = download("pymc-minecraft-llm-kit.zip", "/kaggle/working/pymc-minecraft-llm-kit.zip")
        assert kit, "No data: add your llm_data dataset with Add Input, or turn Internet on to download the kit"
        with zipfile.ZipFile(kit) as archive:
            archive.extractall("/kaggle/working/kit")
        os.remove(kit)
        DATA = "/kaggle/working/kit/llm_data"
assert os.path.isfile(f"{DATA}/train.jsonl"), f"{DATA} has no train.jsonl - run: python -m pymc_bot llm build"
pictures = len(glob.glob(f"{DATA}/images/*"))
print(f"OK    data: {DATA} ({pictures} pictures)")

def trainer_is_current(path):
    text = open(path, encoding="utf-8").read()
    return all(flag in text for flag in ("--uncensored", "--system-prompt-file", "--test-only", "--general",
                                             "--stream-vision", "--minecraft-qa", "--commercial",
                                             "--safetensors", "--gguf-quants", "--deadline", "--export-lock"))

SCRIPT = f"{DATA}/train_minecraft_llm.py" if os.path.isfile(f"{DATA}/train_minecraft_llm.py") else ""
if TRAINER == "latest" or (TRAINER == "auto" and not (SCRIPT and trainer_is_current(SCRIPT))):
    if SCRIPT:
        print("the trainer in your dataset is older than this notebook - fetching the newest one")
    SCRIPT = download("train_minecraft_llm.py", "/kaggle/working/train_minecraft_llm.py") or SCRIPT
assert SCRIPT, "No train_minecraft_llm.py: rebuild your data with python -m pymc_bot llm build, or turn Internet on"
check(trainer_is_current(SCRIPT), f"trainer: {SCRIPT}",
      f"trainer {SCRIPT} is old - UNCENSORED / tests / safetensors need a newer one (rebuild the data)")

# Hugging Face token from Kaggle's secrets (never printed)
if HF_REPO and not QUICK_TEST:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("OK    HF_TOKEN loaded - finished editions are uploaded to Hugging Face")
    except Exception as exc:
        HF_REPO = ""
        print(f"WARN  no HF_TOKEN secret ({type(exc).__name__}) - models stay in /kaggle/working (Output tab)")

# the system prompt goes to the trainer as a file (no shell quoting problems)
SYSTEM_FILE = "/kaggle/working/system_prompt.txt"
with open(SYSTEM_FILE, "w", encoding="utf-8") as handle:
    handle.write(SYSTEM_PROMPT.strip())

FLAGS = f'--data "{DATA}" --system-prompt-file "{SYSTEM_FILE}" --persona-mix {PERSONA_MIX} --epochs {EPOCHS}'
FLAGS += " --uncensored" if UNCENSORED else ""
FLAGS += " --commercial" if COMMERCIAL_ONLY else ""
try:  # the session started when this kernel did
    import psutil
    SESSION_START = psutil.Process(os.getpid()).create_time()
except Exception:
    SESSION_START = time.time()
DEADLINE = SESSION_START + MAX_HOURS * 3600
FLAGS += f" --deadline {DEADLINE:.0f} --export-lock /tmp/pymc-export.lock"
FLAGS += " --clean-after-upload" if CLEAN_AFTER_UPLOAD else ""
FLAGS += " --group-by-length --loader-workers 2 --eval-limit 60" if FAST else ""
if QUICK_TEST:
    FLAGS += f" --max-steps 30 --eval-limit 10 --gguf none --safetensors none --general {min(GENERAL_CHAT, 200)}"
    FLAGS += f" --minecraft-qa {min(MINECRAFT_QA, 200)} --stream-vision {min(STREAM_PICTURES, 40)}"
else:
    FLAGS += f" --general {GENERAL_CHAT} --minecraft-qa {MINECRAFT_QA} --stream-vision {STREAM_PICTURES}"
    FLAGS += " --safetensors 16bit" if SAFETENSORS else " --safetensors none"
    if HF_REPO:
        FLAGS += f' --hub-repo "{HF_REPO}"' + (" --hub-private" if HF_PRIVATE else "") + (" --hub-lora" if HF_LORA else "")
        FLAGS += f' --gguf-quants "{GGUF_QUANTS or "none"}"'
    else:
        FLAGS += " --gguf-quants none"  # without an upload only the main GGUF is kept (Kaggle's 20 GB)
    if TEST_GGUF:
        FLAGS += " --test-gguf"
FLAGS += " --test" if RUN_TESTS else ""
FLAGS += f" {EXTRA}"
print("trainer flags:", FLAGS)

In [ ]:
# ============================== dry run (no GPU time wasted on broken data) ==============================
import shlex

READY = []
for edition in EDITIONS:
    command = ["python", SCRIPT, "--preset", edition, "--dry-run"] + shlex.split(FLAGS)
    check_run = subprocess.run(command, capture_output=True, text=True)
    print(f"===== {edition}\n{check_run.stdout[-3000:]}{check_run.stderr[-1500:]}")
    if check_run.returncode == 0:
        READY.append(edition)
    else:
        print(f"skipping {edition}: the dry run failed (see above)")
print("will train:", READY)

In [ ]:
# ============================== optional: Ollama for the GGUF test ==============================
if TEST_GGUF and not QUICK_TEST and not shutil.which("ollama"):
    !apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    print("Ollama:", shutil.which("ollama") or "install FAILED - the GGUF test will be skipped")

In [ ]:
# ============================== train on both T4s (+ tests, safetensors, GGUF quants, upload) ==============================
# GPU_MODE "auto": 2+ editions -> one edition per T4 at the same time; 1 edition -> both T4s train it together (DDP).
# Each edition logs to /kaggle/working/<edition>.log; the lines are shown here as they come, prefixed with the edition.
import sys
import threading
import time

n_gpus = len(gpus)
MODE = GPU_MODE if n_gpus >= 2 else "single"
if MODE == "auto":
    MODE = "parallel" if len(READY) >= 2 else "ddp"
SLOTS = {"parallel": [[0], [1]], "ddp": [[0, 1]]}.get(MODE, [[0]])
RUN_FLAGS = FLAGS + f" --reserve-minutes {90 if MODE == 'parallel' else 60}"  # parallel: the other export may run first
print(f"{n_gpus} GPU(s), mode {MODE}: " + {"parallel": "one edition per T4, two at a time",
                                           "ddp": "both T4s train each edition together",
                                           "single": "one T4, one edition after another"}.get(MODE, MODE))
if MODE == "parallel" and not HF_REPO and SAFETENSORS:
    print("WARN  two editions without a Hugging Face upload can pass Kaggle's 20 GB output limit "
          "(~11 GB each) - add HF_TOKEN, set SAFETENSORS = False, or train one edition per session")


def train_command(edition, gpu_ids):
    script = [SCRIPT, "--preset", edition] + shlex.split(RUN_FLAGS)
    if len(gpu_ids) > 1:  # DDP: one process per T4, Unsloth syncs the gradients
        return [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={len(gpu_ids)}"] + script
    return [sys.executable] + script


def run_edition(edition, gpu_ids):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": ",".join(map(str, gpu_ids)), "PYTHONUNBUFFERED": "1"}
    with open(f"/kaggle/working/{edition}.log", "a", encoding="utf-8") as log:
        log.write(f"===== {edition} on GPU {','.join(map(str, gpu_ids))} =====\n")
        log.flush()
        return subprocess.run(train_command(edition, gpu_ids), stdout=log, stderr=subprocess.STDOUT, env=env).returncode


def note(edition, text):
    """A line in the edition's log, so it shows up here in order with the trainer's own output."""
    with open(f"/kaggle/working/{edition}.log", "a", encoding="utf-8") as log:
        log.write(text + "\n")


def report_of(edition):
    path = f"/kaggle/working/{edition}/pymc_model.json"
    return json.load(open(path)) if os.path.exists(path) else {}


def clean_up(edition):
    info = report_of(edition)
    if CLEAN_AFTER_UPLOAD and info.get("hub_repo"):
        # GGUF repo is up: drop the GGUF files; the safetensors folder only once its own upload worked too
        big = glob.glob(f"/kaggle/working/{edition}/*.gguf") + [f"/kaggle/working/{edition}/lora"]
        if info.get("safetensors_uploaded") or not info.get("safetensors"):
            big.append(f"/kaggle/working/{edition}/safetensors")
        for path in big:
            if os.path.isdir(path):
                shutil.rmtree(path, ignore_errors=True)
            elif os.path.exists(path):
                os.remove(path)
        note(edition, "uploaded, big local files cleaned up")
    if info.get("safetensors") and not info.get("safetensors_uploaded"):
        note(edition, f"the safetensors edition is in /kaggle/working/{edition}/safetensors (not uploaded)")


queue, queue_lock, CODES = list(READY), threading.Lock(), {}


def worker(gpu_ids):
    while True:
        with queue_lock:
            if not queue:
                return
            edition = queue.pop(0)
        code = run_edition(edition, gpu_ids)
        if code not in (0, 3) and len(gpu_ids) > 1 and not report_of(edition):
            note(edition, f"training on both T4s failed (exit {code}) - training it again on one T4")
            code = run_edition(edition, gpu_ids[:1])
        CODES[edition] = code
        clean_up(edition)


threads = [threading.Thread(target=worker, args=(slot,), daemon=True) for slot in SLOTS]
for thread in threads:
    thread.start()
offsets, partial, progress, last_beat = {}, {}, {}, time.time()
while True:
    alive = any(thread.is_alive() for thread in threads)
    for edition in READY:
        path = f"/kaggle/working/{edition}.log"
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8", errors="replace") as handle:
            handle.seek(offsets.get(edition, 0))
            chunk = handle.read()
            offsets[edition] = handle.tell()
        text = partial.get(edition, "") + chunk
        *lines, partial[edition] = text.split("\n")
        for line in lines:
            line = line.split("\r")[-1].rstrip()
            if line:
                print(f"[{edition}] {line}", flush=True)
        if "\r" in partial[edition]:  # a progress bar: shown every few minutes, not on every step
            progress[edition] = partial[edition].split("\r")[-1].strip() or progress.get(edition, "")
    if time.time() - last_beat > 180:
        last_beat = time.time()
        left = (DEADLINE - time.time()) / 3600
        for edition, bar in progress.items():
            if bar and edition not in CODES:
                print(f"[{edition}] {bar}   (session: {left:.1f} h left)", flush=True)
    if not alive:
        break
    time.sleep(5)
for edition in READY:
    code = CODES.get(edition)
    print(f"{edition}: " + {0: "done", 3: "skipped - not enough session time left (train it next session)"}.get(
        code, f"FAILED (exit {code}) - see /kaggle/working/{edition}.log"))


In [ ]:
# ============================== results ==============================
for edition in READY:
    folder = f"/kaggle/working/{edition}"
    if not os.path.exists(f"{folder}/pymc_model.json"):
        print(f"{edition}: no report - look at the training output above")
        continue
    info = json.load(open(f"{folder}/pymc_model.json"))
    where = f"https://huggingface.co/{info['hub_repo']}" if info.get("hub_repo") else folder
    print(f"\n### {info.get('model_name', edition)} -> {where}")
    print(f"loss {info.get('loss')} | {info.get('train_rows')} rows ({info.get('general_rows', 0)} everyday) | "
          f"{info.get('minutes')} min | eval {info.get('eval', {})}")
    if info.get("stopped_early"):
        print(f"stopped early at step {info['stopped_early']['step']}/{info['stopped_early']['of']} (session time limit)")
    if info.get("tests"):
        print("tests:", info["tests"])
    if info.get("gguf_test"):
        print("GGUF test:", "ok" if info["gguf_test"].get("ok") else info["gguf_test"])
    if info.get("safetensors_uploaded"):
        print(f"safetensors ({info.get('safetensors_gb')} GB): https://huggingface.co/{info['hub_repo_safetensors']}")
    elif info.get("safetensors"):
        print(f"safetensors ({info.get('safetensors_gb')} GB): {folder}/safetensors")
    if info.get("ggufs"):
        print("GGUFs:", ", ".join(f"{g['quant']} {g['gb']} GB" + ("" if g.get("uploaded") or g["quant"] == info.get("quant")
                                                                   else " (local)") for g in info["ggufs"]))
    if info.get("params_b"):
        print(f"parameters: {info['params_b']}B (compact: {info.get('compact', {}).get('tied_dropped') or 'nothing to drop'})")
    if info.get("hub_repo"):
        print(f"on your PC:  python -m pymc_bot llm import --hf {info['hub_repo']}  (add --quant Q8_0 for the big one)")

## Optional: test again / chat with a trained edition

Needs the edition's `lora` folder in `/kaggle/working/<edition>` (set `CLEAN_AFTER_UPLOAD = False` before training to keep it).

In [ ]:
RETEST = []   # e.g. ["minecraftism"]: run the test suite again on the saved adapter
for edition in RETEST:  # noqa: B007 - used by the ! line
    !python "{SCRIPT}" --preset {edition} --test-only {FLAGS}

In [ ]:
CHAT_WITH = ""   # e.g. "minecraftism": chat with a text edition right here (empty line = stop). Not for "Save & Run All".
if CHAT_WITH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(f"/kaggle/working/{CHAT_WITH}/lora", max_seq_length=4096,
                                                         load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    history = [{"role": "system", "content": SYSTEM_PROMPT.strip()}]
    while True:
        line = input("you> ").strip()
        if not line:
            break
        history.append({"role": "user", "content": line})
        prompt = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text=prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        output = model.generate(**inputs, max_new_tokens=1024, temperature=0.7, top_p=0.8, top_k=20, do_sample=True)
        reply = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        print(f"minecraftism> {reply}\n")
        history.append({"role": "assistant", "content": reply.split("</think>")[-1].strip()})
    del model
    import torch
    torch.cuda.empty_cache()

## Next
* From Hugging Face: `python -m pymc_bot llm import --hf <your-hf-name>/Minecraftism-4B-Uncensored-GGUF`
  (creates the Ollama model with this system prompt, e.g. `minecraftism-uncensored`, and switches the bot to it).
  Or download `/kaggle/working/<edition>` from the Output tab and run `python -m pymc_bot llm import <folder>`.
* The **safetensors** edition (`<your-hf-name>/Minecraftism-4B-Uncensored`, no `-GGUF`) is for transformers / vLLM
  (`vllm serve <your-hf-name>/Minecraftism-4B-Uncensored`), further fine-tuning or your own quantisations; its model
  card has the code. `llm import --hf` on it uses the GGUF twin automatically.
* Other GGUF sizes from the same repo: `ollama run hf.co/<your-hf-name>/Minecraftism-4B-Uncensored-GGUF:Q8_0`
  (or `:Q6_K`, `:Q5_K_M`, `:Q3_K_M`); `llm import --hf ... --quant Q8_0` does the same for the bot.
* As a daily assistant without the bot: `ollama run minecraftism-uncensored` (or any Ollama app such as Open WebUI).
* VL editions also turn on **let the model see** in the bot (optional vision packages, see docs/MINECRAFT_LLM.md).
* Compare: `python -m pymc_bot llm eval --model minecraftism-uncensored` vs `--model qwen3:4b`.

In [ ]:
# ============================== auto exit: end the session when everything is done ==============================
# Kaggle has no "stop session" call, so:
# * Save Version -> "Save & Run All" (recommended for long runs): the session ends by itself after this last cell
#   and /kaggle/working is kept as the notebook's Output - nothing to do here.
# * Interactive editor: the session would idle on (and use GPU quota) until Kaggle's idle timeout. When every
#   trained edition is safely on Hugging Face, this stops the kernel and its processes; files that only exist in
#   /kaggle/working are never thrown away like that (they vanish with an interactive session).
run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
reports = {edition: report_of(edition) for edition in READY}
on_hub = bool(reports) and all(info.get("hub_repo") and (info.get("safetensors_uploaded") or not info.get("safetensors"))
                               for info in reports.values())
if not AUTO_EXIT:
    print("AUTO_EXIT = False - remember to stop the session (Run > Stop session) to save GPU quota")
elif run_type == "Batch":
    print("Save & Run All: the session ends right after this cell; /kaggle/working is saved as the Output")
elif on_hub:
    print("everything is on Hugging Face - ending this session now so no GPU time is wasted")
    print("(if the editor still shows it as running, press Run > Stop session)", flush=True)
    time.sleep(5)
    subprocess.Popen(["bash", "-c", "sleep 1; kill -9 -1"], start_new_session=True)  # every process of this session
    os._exit(0)
else:
    print("not ending the session: some files are only in /kaggle/working (no upload, or an upload failed) - "
          "download them from the Output / file panel, then press Run > Stop session")
